In [1]:
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

ytt_api = YouTubeTranscriptApi()

In [3]:
video_id = 'syE8f_ow0iM'
print(ytt_api.list(video_id=video_id))

For this video (syE8f_ow0iM) transcripts are available in the following languages:

(MANUALLY CREATED)
None

(GENERATED)
 - en ("English (auto-generated)")[TRANSLATABLE]
 - fr-FR ("French (France) (auto-generated)")[TRANSLATABLE]
 - de-DE ("German (Germany) (auto-generated)")[TRANSLATABLE]
 - hi ("Hindi (auto-generated)")[TRANSLATABLE]
 - id ("Indonesian (auto-generated)")[TRANSLATABLE]
 - it ("Italian (auto-generated)")[TRANSLATABLE]
 - pt-BR ("Portuguese (Brazil) (auto-generated)")[TRANSLATABLE]
 - es-US ("Spanish (United States) (auto-generated)")[TRANSLATABLE]

(TRANSLATION LANGUAGES)
 - ar ("Arabic")
 - zh-Hant ("Chinese (Traditional)")
 - nl ("Dutch")
 - en ("English")
 - fr ("French")
 - de ("German")
 - hi ("Hindi")
 - id ("Indonesian")
 - it ("Italian")
 - ja ("Japanese")
 - ko ("Korean")
 - pt ("Portuguese")
 - ru ("Russian")
 - es ("Spanish")
 - th ("Thai")
 - uk ("Ukrainian")
 - vi ("Vietnamese")


In [4]:
try:
    transcript_list = ytt_api.fetch(video_id=video_id)
    print('Transcript list length: ', len(transcript_list), end='\n\n')
    print(transcript_list[0])
except TranscriptsDisabled:
    print('Transcript disabled for this video')

Transcript list length:  2404

FetchedTranscriptSnippet(text='What is up ladies and gentlemen? Welcome', start=0.24, duration=4.32)


In [5]:
transcript = ' '.join(chunk.text for chunk in transcript_list)
print('Total words in the transcript: ', len(transcript))

Total words in the transcript:  86691


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

In [7]:
chunks = splitter.create_documents(texts=[transcript])
print('Number of chunks: ', len(chunks), end='\n\n')
print('First chunk: ', chunks[0])


Number of chunks:  134

First chunk:  page_content='What is up ladies and gentlemen? Welcome back to Fudge Muppet. I'm Scott here with Michael and today we're talking all the law you need to know for Elder Scrolls 6. >> Yes. And I have a feeling this is going to be a series. Yeah. Series that people have been wanting for a long time. Dare I say the return of the Elder Scrolls 6 podcast. Maybe potentially probably. Now, we've been waiting for Elder Scrolls 6 for a very, very, very long time. In fact, since high school. We made this channel when we were wearing school uniforms and we would joke about how we'll have kids playing The Elder Scrolls 6. Well, I can tell you what, we need to wear glasses for this episode. >> We're not going to be able to read >> cringe glasses reveal, but we literally need to wear glasses. We have gray hairs on the'


___
Creating vector store:

In [51]:
from langchain_openai import OpenAIEmbeddings

embedder = OpenAIEmbeddings(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",     # type: ignore
    model="local-bge-m3",
    check_embedding_ctx_length=False
)

In [52]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedder,
    collection_name='clearer_notebook_collection'
)



Indexing completed, Retrieval next

In [53]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={
        'k':10
    }
)

Augmentation

In [60]:
from langchain_core.prompts import PromptTemplate

llm_prompt_template = PromptTemplate(
    template="""
You are a question-answering assistant.

Answer the question using ONLY the information contained in the CONTEXT below.

IMPORTANT:
- Do not use your own knowledge.
- Do not assume facts that are not present in the context.
- If the context does not contain enough information to answer the question, say "I don't know based on the provided context."
- The phrase "Great War" may refer to something specific in the provided context. Use the meaning given by the context, not your general knowledge.
- Do not provide information about World War I unless the context explicitly discusses World War I.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER:
""",
    input_variables=["context", "question"],
    validate_template=True
)

In [62]:
user_query = input('Ask about things in the video')

retrieved_docs = retriever.invoke(user_query)
context = ' '.join(chunk.page_content for chunk in retrieved_docs)

final_prompt = llm_prompt_template.invoke({'context': context, 'question':user_query})

Next, generation:

In [63]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model='local qwen3.5:4b-8b',
    base_url='http://127.0.0.1:8080/v1',
    api_key='not needed' # type: ignore
)

In [64]:
result = llm.invoke(user_query)
print(result.content)

</think>

# The Great War (World War I)

The Great War, more commonly known as **World War I**, was a global conflict fought between 1914 and 1918. It is widely considered the first "total war" in history, marking a definitive turning point in human history where nations mobilized their entire economies and populations to wage a prolonged conflict.

Here is a comprehensive overview of its origins, key events, and lasting impact.

---

### 1. The Dates and Scope
*   **Start:** July 28, 1914 (Austro-Hungarian declaration of war on Serbia) / August 1, 1914 (Germany declared war on France).
*   **End:** November 11, 1918 (Armistice signed).
*   **Participants:** Approximately 30 countries involved.
*   **Casualties:** Roughly 20 million military dead and 6 million civilian dead (approx. 18% of the world's population at the time).

### 2. The Main Conflict Sides
The war was fought between two main alliances:

*   **The Central Powers:**
    *   Germany (the main aggressor in Europe)
    *  

In [59]:
retrieved_docs

[Document(id='645cab47-6967-4f5f-b9b3-5363091ee887', metadata={}, page_content="but anyway so we should all be pretty familiar with the great war so it starts off towards the end of the fourth era 171 and it's basically the old Mary Dominion show up to Siridil and they're like to the emperor and we're like we killed all of your blades agents within all of the uh all of Veilenwood and any old Mary Dominion territories and Uh, guess what? Give us the south of Hammerfell. Stop worshipping Talos. Give us a ton of money. Just do it. Like straight up this Titus meme says the second says no. And then so the Great War begins. And it really starts off with, and we can read the the Great War book talking about it. It starts off as if the original goal of the Old Mary Dominion was kind of just to take Hammerfell and they sent a force into the uh southwest of Siridel around"),
 Document(id='d265fba2-95ea-44a3-acbb-0e835603dc3b', metadata={}, page_content="on there's all these different moving piec